# BERTopic Trial
Eksperimen topic modelling untuk melihat apakah BERTopic berhasil menangkap topik. Sesuaikan `DATA_PATH`, `TEXT_COL`, `DATE_COL` di sel 2.

## 0. Install (sekali saja, lalu restart kernel)

In [1]:
%pip install -U sentence-transformers bertopic umap-learn hdbscan gensim plotly

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ------- -------------------------------- 1.8/9.7 MB 16.9 MB/s eta 0:00:01
   ------------------------ --------------- 6.0/9.7 MB 18.5 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 18.9 MB/s  0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 6.5.1
    Uninstalling plotly-6.5.1:
      Successfully uninstalled plotly-6.5.1
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Setup

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.io as pio
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import MaximalMarginalRelevance

pio.renderers.default = "notebook"
SEED = 42
np.random.seed(SEED)

c:\Users\grace\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load data + cleaning minimal

In [4]:
data_candidates = [
    Path("data/processed/merged_data_clean.csv"),
    Path("../data/processed/merged_data_clean.csv"),
]
DATA_PATH = next((path for path in data_candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Tidak menemukan data/processed/merged_data_clean.csv. "
        "Jalankan notebook dari root proyek atau folder percobaan task 3."
    )

TEXT_COL = "all_text_lemmatized"
DATE_COL = "tanggal"

df = pd.read_csv(DATA_PATH)
missing_columns = {column for column in [TEXT_COL, DATE_COL] if column not in df.columns}
if missing_columns:
    raise KeyError(f"Kolom tidak ditemukan: {sorted(missing_columns)}")
df = df.dropna(subset=[TEXT_COL]).drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)
df[TEXT_COL] = df[TEXT_COL].astype(str).str.replace(r"<.*?>|http\S+", " ", regex=True).str.strip()
df = df[df[TEXT_COL].str.split().str.len() >= 5].reset_index(drop=True)

docs = df[TEXT_COL].tolist()
timestamps = pd.to_datetime(df[DATE_COL]).tolist() if DATE_COL else None
print(f"Jumlah dokumen: {len(docs)}")

FileNotFoundError: [Errno 2] No such file or directory: 'data.csv'

## 3. Embedding (Freeze Weights, Tanpa Cache Disk)


In [ ]:
import torch
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
for param in embedding_model.parameters():
    param.requires_grad = False
embedding_model.eval()

with torch.no_grad():
    embeddings = embedding_model.encode(docs, show_progress_bar=True, batch_size=64)
print("Embeddings shape (in-memory, no cache):", embeddings.shape)


## 4. Konfigurasi & Hyperparameter Tuning BERTopic (`min_cluster_size`, `n_neighbors`)


In [ ]:
try:
    from gensim.corpora import Dictionary
    from gensim.models.coherencemodel import CoherenceModel
    HAS_GENSIM = True
except ImportError:
    HAS_GENSIM = False

vectorizer_model = CountVectorizer(stop_words="english", min_df=3, ngram_range=(1, 2))
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
representation_model = MaximalMarginalRelevance(diversity=0.3)

# Gunakan matriks biner sparse CountVectorizer (hemat RAM, bebas MemoryError)
X_coh_bin = (vectorizer_model.fit_transform(docs) > 0).tocsc()
voc_coh = vectorizer_model.vocabulary_
n_docs_coh = max(X_coh_bin.shape[0], 1)

def compute_coherence_score(topic_words_list):
    if not topic_words_list:
        return 0.0
    if HAS_GENSIM:
        analyzer = vectorizer_model.build_analyzer()
        tokens_tune = [analyzer(d) for d in docs]
        dictionary_tune = Dictionary(tokens_tune)
        return float(CoherenceModel(topics=topic_words_list, texts=tokens_tune, dictionary=dictionary_tune, coherence="c_v", processes=1).get_coherence())
    scores = []
    for ws in topic_words_list:
        cols = [voc_coh[w] for w in ws[:10] if w in voc_coh]
        for i in range(len(cols)):
            ci = X_coh_bin[:, cols[i]]
            d_i = int(ci.nnz)
            if d_i == 0:
                continue
            for j in range(i + 1, len(cols)):
                cj = X_coh_bin[:, cols[j]]
                d_j = int(cj.nnz)
                if d_j == 0:
                    continue
                d_ij = int(ci.multiply(cj).nnz)
                if d_ij > 0:
                    pmi = np.log((d_ij * n_docs_coh) / (d_i * d_j))
                    npmi = pmi / (-np.log(d_ij / n_docs_coh) + 1e-9)
                    scores.append(npmi)
    return float(np.mean(scores)) if scores else 0.0

mcs_candidates = sorted(list({max(5, len(docs) // 300), max(8, len(docs) // 200), max(12, len(docs) // 120)}))
cand_pairs = [(10, mcs_candidates[0]), (15, mcs_candidates[len(mcs_candidates)//2]), (15, mcs_candidates[-1]), (20, mcs_candidates[len(mcs_candidates)//2])]
cand_pairs = list(dict.fromkeys(cand_pairs))

tuning_logs = []
best_score = -1e9
best_cfg = {"n_neighbors": 15, "min_cluster_size": mcs_candidates[0]}

for nn_val, mcs in cand_pairs:
    umap_cand = UMAP(n_neighbors=nn_val, n_components=5, min_dist=0.0, metric="cosine", random_state=SEED)
    hdb_cand = HDBSCAN(min_cluster_size=mcs, min_samples=max(3, mcs // 2), metric="euclidean", cluster_selection_method="eom", prediction_data=True)
    tm_cand = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_cand,
        hdbscan_model=hdb_cand,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ctfidf_model,
        representation_model=representation_model,
        top_n_words=10,
        verbose=False,
    )
    t_cand, _ = tm_cand.fit_transform(docs, embeddings)
    t_ids = [t for t in tm_cand.get_topics() if t != -1]
    out_ratio = float((np.array(t_cand) == -1).mean())
    if len(t_ids) >= 2:
        t_words = [[w for w, _ in tm_cand.get_topic(t)] for t in t_ids]
        coh = compute_coherence_score(t_words)
    else:
        coh = 0.0
    score = coh - 0.15 * out_ratio
    tuning_logs.append({"n_neighbors": nn_val, "min_cluster_size": mcs, "n_topics": len(t_ids), "outlier_pct": round(out_ratio * 100, 2), "coherence": round(coh, 4), "tune_score": round(score, 4)})
    if score > best_score:
        best_score = score
        best_cfg = {"n_neighbors": nn_val, "min_cluster_size": mcs}

tuning_df = pd.DataFrame(tuning_logs).sort_values("tune_score", ascending=False).reset_index(drop=True)
print("Hasil Hyperparameter Tuning BERTopic:")
display(tuning_df)
print("Best Hyperparameters:", best_cfg)

min_cluster_size = best_cfg["min_cluster_size"]
umap_model = UMAP(n_neighbors=best_cfg["n_neighbors"], n_components=5, min_dist=0.0, metric="cosine", random_state=SEED)
hdbscan_model = HDBSCAN(min_cluster_size=min_cluster_size, min_samples=max(3, min_cluster_size // 2), metric="euclidean", cluster_selection_method="eom", prediction_data=True)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    top_n_words=10,
    verbose=True,
)


## 5. Training

In [ ]:
topics, probs = topic_model.fit_transform(docs, embeddings)
topics_raw = list(topics)

info = topic_model.get_topic_info()
n_topics = (info["Topic"] != -1).sum()
n_out = (np.array(topics) == -1).sum()
print(f"Topik: {n_topics} | Outlier: {n_out} ({n_out/len(docs)*100:.1f}%)")
info.head(15)

## 6. Reduksi outlier

In [ ]:
if n_out > 0:
    new_topics = topic_model.reduce_outliers(
        docs, topics, strategy="embeddings", embeddings=embeddings, threshold=0.3
    )
    topic_model.update_topics(
        docs,
        topics=new_topics,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ctfidf_model,
        representation_model=representation_model,
    )
    topics = new_topics
    left = (np.array(topics) == -1).sum()
    print(f"Outlier sisa: {left} ({left/len(docs)*100:.1f}%)")

topic_model.get_topic_info().head(15)

## 7. Evaluasi: apakah topiknya bermakna

In [ ]:
topic_ids = [t for t in topic_model.get_topics() if t != -1]
topic_words = [[w for w, _ in topic_model.get_topic(t)] for t in topic_ids]

coherence = compute_coherence_score(topic_words)
diversity = len({w for ws in topic_words for w in ws}) / max(sum(len(ws) for ws in topic_words), 1)
lbl_coh = "Coherence (c_v)" if HAS_GENSIM else "Coherence (NPMI)"

print(f"{lbl_coh}: {coherence:.3f}")
print(f"Topic diversity: {diversity:.3f}   (mendekati 1 = topik tidak tumpang tindih)")

# Inspeksi manual: 3 dokumen representatif per topik
for t in topic_ids[:10]:
    words = ", ".join(w for w, _ in topic_model.get_topic(t)[:6])
    print(f"\n=== Topik {t} | {words}")
    for d in topic_model.get_representative_docs(t)[:3]:
        print("  -", d[:160].replace("\n", " "))


## 8. Visualisasi

In [ ]:
topic_model.visualize_barchart(top_n_topics=8, n_words=8).show()

if n_topics >= 5:
    topic_model.visualize_topics().show()

topic_model.visualize_heatmap().show()
topic_model.visualize_hierarchy().show()

## 9. Topik terhadap waktu (hanya jika ada tanggal)

In [ ]:
if timestamps is not None:
    tot = topic_model.topics_over_time(docs, timestamps, nr_bins=20)
    topic_model.visualize_topics_over_time(tot, top_n_topics=6).show()